# Short-Term Memory

_Impementing a short-term memory for an agent._

 Integrates a standard in-memory checkpoint saver into an agent to remember previous interactions and maintain context during a conversation resulting increased user experience.

In [1]:
# Imports packages

from langchain_ollama.chat_models import ChatOllama
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
# Sets endpoints for Ollama models to be available over web requests.

OLLAMA_MODEL = "llama3.2:3b"
OLLAMA_ENDPOINT = "http://localhost:11434/"

In [3]:
# Initializes chat client
chat_model = ChatOllama(
    model=OLLAMA_MODEL,
    reasoning=False,
    base_url=OLLAMA_ENDPOINT)

In [4]:
# Creates an agent

agent = create_agent(
    model=chat_model,               # The language model for the agent.
    checkpointer=InMemorySaver(),   # An in-memory checkpoint saver to store checkpoints in memory 
                                    # Suitable only for testing & debugging purposes. Production system should use persistent memory providers.
    system_prompt="""You are a helpful intelligent assistant. Keep responses short, clear, 
        and meaningful (about 1-3 sentences).""",
)

In [ ]:
# Fixed thread_id ensures short-term memory persists across turns in this chat session
config = {"configurable": {"thread_id": "memory-demo-thread"}}


def stream_agent_reply(user_text: str) -> None:
    """
    Streams agent's reply to the console as it is generated, token by token.
    
    Parameters:
    user_text (str): The input message from the user to which the agent will respond.
    """

    print("Agent: ", end="")

    for part in agent.stream(
            {"messages": [{"role": "user", "content": user_text}]},
            config=config,
            stream_mode="messages",     # To stream language model outputs token by token 
            version="v2"                # v2 to get a unified output format where v1 output is streaming mode specific
    ):
        message_chunk, metadata = part["data"]              # Extracts chunk that contains the human-readable message.
        print(message_chunk.content, end="", flush=True)

    print()

In [ ]:
# Agent-user conversation loop

WECOME_MESSAGE = "Agent: Hi! I am your assistant. Tell me something about yourself.\n"

print(WECOME_MESSAGE)

while True:                                 # Loops over until user enters 'exit' or 'quit'
    user_input = input("You: ").strip()     # Receives user message
    if not user_input:                      # Ensures user enters a non-empty message
        print("Agent: Please type a message, or enter 'exit' to quit.")
        continue

    print(f"You: {user_input}\n")               # Prints user message

    
    if user_input.lower() in {"exit", "quit"}:  # Exits conversation loop if user enters `exit` or `quit`
        print("Agent: Goodbye!")
        break

    stream_agent_reply(user_input)          # Streams agent's response

Agent: Hi! I am your assistant. Tell me something about yourself.

You: Hi! My name is Bob.

Agent: Hello Bob! It's nice to meet you. I'm here to help answer any questions or provide assistance with anything you'd like to discuss. How can I start helping you today?
You: I am interested in Electronics. Can you please suggest me few of the 'do-it-yourself' projects?

Agent: Bob, that's a great hobby! Here are some fun and easy DIY electronics projects to get you started: 

1. Building an LED Circuit
2. Creating a Simple Arduino Project (e.g., Blinking LED)
3. Making a Basic Robot using an Arduino and Motors
4. Building a Simple Radio Transmitter
5. Creating a DIY Home Automation System with Sensors and Relays

Which one of these projects sounds interesting to you?
You: The third projects seems to be an interesting one. Can you please provide bill of materials or little more details for that?

Agent: For the DIY Robot project using Arduino, here's a simplified Bill of Materials (BOM):

**